# **Exploratory Data Analysis**

This notebook performs exploratory analysis on the reduced EEG dataset.  
It focuses on evaluating feature distributions, class separability, and condition-driven effects to determine whether the data contains meaningful signals for schizophrenia detection.

### **Import Libraries**

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd
import numpy as np

from IPython.display import Markdown, display

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### **Initialize Spark Session**

In [2]:
spark = (
    SparkSession.builder
    .appName("EEG_EDA_Reduced_Data")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 22:18:30 WARN Utils: Your hostname, Sarras-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.9 instead (on interface en0)
26/04/13 22:18:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 22:18:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 22:18:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### **Define Paths**

In [3]:
project_root = Path("..").resolve()

reduced_data_dir = project_root / "data" / "processed" / "all_csv_agg_ALLLL"
demographic_path = project_root / "data" / "demographic.csv"

### **Define Plot Theme and Color Palette**

In [4]:
PLOTLY_TEMPLATE = "plotly_white"

GENERAL_PALETTE = [
    "#5A7ACD",
    "#2FA084",
    "#F68048",
]

CLASS_COLOR_MAP = {
    "Control": "#3852B4",
    "Schizophrenia": "#D84B4B",
}

LABEL_NAME_MAP = {
    0: "Control",
    1: "Schizophrenia",
}

CONDITION_NAME_MAP = {
    1: "Active",
    2: "Passive",
    3: "Control",
}

def apply_layout(fig, title, x_title=None, y_title=None):
    fig.update_layout(
        template=PLOTLY_TEMPLATE,

        title=dict(
            text=f"<b>{title}</b>",
            x=0.05,              
            xanchor="left",
            font=dict(color="black")
        ),

        font=dict(
            family="Arial",
            size=14,
            color="black"
        ),

        paper_bgcolor="white",
        plot_bgcolor="white",

        xaxis_title=x_title,
        yaxis_title=y_title,

        legend_title_text="",

        margin=dict(l=40, r=40, t=70, b=40)
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        color="black"
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
        color="black"
    )

    return fig

### **Load the Reduced EEG Table and Demographic Table**

In [5]:
reduced_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(reduced_data_dir))
)

demographic_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(demographic_path))
)

demographic_df = demographic_df.toDF(*[c.strip() for c in demographic_df.columns])
demographic_df = demographic_df.dropDuplicates(["subject"])

### **Prepare the Label Table and Join it to the Reduced EEG Data**

In [6]:
label_df = (
    demographic_df
    .select("subject", F.col("group").alias("label"), "gender", "age", "education")
)

eda_df = reduced_df.join(label_df, on="subject", how="left")

### **Dataset Overview**

In [7]:
n_rows = eda_df.count()
n_cols = len(eda_df.columns)
n_subjects = eda_df.select("subject").distinct().count()
n_trials = eda_df.select("subject", "trial").distinct().count()

print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")
print(f"Unique subjects: {n_subjects}")
print(f"Unique subject-trial pairs: {n_trials}")

Rows: 23201
Columns: 1547
Unique subjects: 81
Unique subject-trial pairs: 8094


### **Inspect Key Columns**

In [8]:
eda_df.select("subject", "trial", "condition", "label", "gender", "age", "education").show(10, truncate=False)

+-------+-----+---------+-----+------+----+---------+
|subject|trial|condition|label|gender|age |education|
+-------+-----+---------+-----+------+----+---------+
|1.0    |1.0  |1.0      |0.0  | M    |44.0|16.0     |
|1.0    |1.0  |2.0      |0.0  | M    |44.0|16.0     |
|1.0    |2.0  |1.0      |0.0  | M    |44.0|16.0     |
|1.0    |2.0  |2.0      |0.0  | M    |44.0|16.0     |
|1.0    |2.0  |3.0      |0.0  | M    |44.0|16.0     |
|1.0    |3.0  |1.0      |0.0  | M    |44.0|16.0     |
|1.0    |3.0  |3.0      |0.0  | M    |44.0|16.0     |
|1.0    |4.0  |1.0      |0.0  | M    |44.0|16.0     |
|1.0    |4.0  |2.0      |0.0  | M    |44.0|16.0     |
|1.0    |4.0  |3.0      |0.0  | M    |44.0|16.0     |
+-------+-----+---------+-----+------+----+---------+
only showing top 10 rows


### **Check for Missing Values in Key Columns**

In [9]:
eda_df.select(
    F.sum(F.col("label").isNull().cast("int")).alias("missing_label"),
    F.sum(F.col("condition").isNull().cast("int")).alias("missing_condition"),
    F.sum(F.col("subject").isNull().cast("int")).alias("missing_subject"),
    F.sum(F.col("trial").isNull().cast("int")).alias("missing_trial")
).show()

+-------------+-----------------+---------------+-------------+
|missing_label|missing_condition|missing_subject|missing_trial|
+-------------+-----------------+---------------+-------------+
|            0|                0|              0|            0|
+-------------+-----------------+---------------+-------------+



### **Summary Statistics for Age and Education**

In [10]:
demographic_df.select("age", "education").describe().show()

+-------+------------------+------------------+
|summary|               age|         education|
+-------+------------------+------------------+
|  count|                81|                81|
|   mean|39.370370370370374|14.487654320987655|
| stddev|13.594525041762626| 2.267949664126685|
|    min|              19.0|               9.0|
|    max|              63.0|              19.0|
+-------+------------------+------------------+



### **Subject-Level Target Distribution**

In [11]:
subject_label_pdf = (
    eda_df
    .select("subject", "label")
    .distinct()
    .groupBy("label")
    .count()
    .orderBy("label")
    .toPandas()
)

subject_label_pdf["label_name"] = subject_label_pdf["label"].map({
    0: "Control",
    1: "Schizophrenia"
})

total_subjects = subject_label_pdf["count"].sum()
subject_label_pdf["percentage"] = (
    subject_label_pdf["count"] / total_subjects * 100
).round(2)

subject_label_pdf

,label,count,label_name,percentage
0,0.0,32,Control,39.51
1,1.0,49,Schizophrenia,60.49


In [ ]:
fig = px.bar(
    subject_label_pdf,
    x="label_name",
    y="percentage",
    color="label_name",
    text=subject_label_pdf["percentage"].astype(str) + "%",
    color_discrete_map=CLASS_COLOR_MAP
)

fig = apply_layout(
    fig,
    "Subject-Level Target Distribution",
    "Class",
    "Percentage"
)

fig.update_traces(textposition="outside", opacity=0.9)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.show()

### **Interpretation**

The dataset shows a moderate class imbalance at the subject level, with approximately 60.5% schizophrenia patients and 39.5% controls.  
While both classes are reasonably represented, this imbalance should be considered when choosing the appropriate evaluation metric to optimize, ensuring the model does not become biased toward the majority class.

### **Trial-Level Target Distribution**

In [13]:
trial_label_pdf = (
    eda_df
    .groupBy("label")
    .count()
    .orderBy("label")
    .toPandas()
)

trial_label_pdf["label_name"] = trial_label_pdf["label"].map({
    0: "Control",
    1: "Schizophrenia"
})

total_trials = trial_label_pdf["count"].sum()
trial_label_pdf["percentage"] = (
    trial_label_pdf["count"] / total_trials * 100
).round(2)

trial_label_pdf

,label,count,label_name,percentage
0,0.0,9226,Control,39.77
1,1.0,13975,Schizophrenia,60.23


In [28]:
fig = px.bar(
    trial_label_pdf,
    x="label_name",
    y="percentage",
    color="label_name",
    text=trial_label_pdf["percentage"].astype(str) + "%",
    color_discrete_map=CLASS_COLOR_MAP
)

fig = apply_layout(
    fig,
    "Trial-Level Target Distribution",
    "Class",
    "Percentage"
)

fig.update_traces(textposition="outside", opacity=0.9)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.show()

### **Interpretation**

At the trial level, the class distribution closely mirrors the subject-level imbalance, with approximately 60.2% of trials belonging to schizophrenia patients and 39.8% to controls. 
 
This consistency indicates that trials are fairly evenly distributed across subjects without introducing additional imbalance, but the overall skew should still be considered when selecting evaluation metrics to ensure balanced model performance.

### **Condition Distribution by Target**

In [17]:
condition_label_pdf = (
    eda_df
    .groupBy("label", "condition")
    .count()
    .orderBy("label", "condition")
    .toPandas()
)

condition_label_pdf["label_name"] = condition_label_pdf["label"].map({
    0: "Control",
    1: "Schizophrenia"
})

condition_label_pdf["percentage"] = (
    condition_label_pdf.groupby("label")["count"]
    .transform(lambda x: x / x.sum() * 100)
).round(2)

condition_label_pdf

,label,condition,count,label_name,percentage
0,0.0,1.0,3108,Control,33.69
1,0.0,2.0,3007,Control,32.59
2,0.0,3.0,3111,Control,33.72
3,1.0,1.0,4728,Schizophrenia,33.83
4,1.0,2.0,4624,Schizophrenia,33.09
5,1.0,3.0,4623,Schizophrenia,33.08


In [26]:
fig = px.bar(
    condition_label_pdf,
    x="condition",
    y="count",
    color="label_name",
    barmode="group",
    text="count",
    color_discrete_map=CLASS_COLOR_MAP
)

fig = apply_layout(
    fig,
    "Condition Distribution by Class",
    "Condition",
    "Number of Trials"
)

fig.update_traces(textposition="outside", opacity=0.9)

fig.update_yaxes(showticklabels=False)

fig.show()

### **Interpretation**

The distribution of trials across conditions is relatively balanced within each class, with similar counts observed for conditions 1, 2, and 3 for both control and schizophrenia groups.  

This indicates that the experimental setup is consistent across conditions and does not introduce additional bias, allowing for fair comparison of EEG responses across different task conditions when analyzing class differences.

### **Create a Compact Plotting Table for Targeted ERP EDA**

In [20]:
FOCUS_ERP_COLS = [
    "Fz_N100_avg",
    "FCz_N100_avg",
    "Cz_N100_avg",
    "Fz_P200_avg",
    "FCz_P200_avg",
    "Cz_P200_avg",
    "Fz_B0_avg",
    "FCz_B0_avg",
    "Cz_B0_avg",
]

available_erp_cols = [c for c in FOCUS_ERP_COLS if c in eda_df.columns]
print(f"ERP columns available for focused EDA: {available_erp_cols}")

eda_plot_pdf = (
    eda_df
    .select(
        "subject",
        "trial",
        "condition",
        "label",
        "gender",
        "age",
        "education",
        *available_erp_cols,
    )
    .toPandas()
)

eda_plot_pdf["label_name"] = eda_plot_pdf["label"].map(LABEL_NAME_MAP)
eda_plot_pdf["condition_name"] = eda_plot_pdf["condition"].map(CONDITION_NAME_MAP).fillna(
    eda_plot_pdf["condition"].astype(str)
)
eda_plot_pdf["gender"] = eda_plot_pdf["gender"].astype(str).str.strip()

subject_demo_pdf = (
    eda_plot_pdf[["subject", "label", "label_name", "gender", "age", "education"]]
    .drop_duplicates(subset=["subject"])
    .copy()
)

condition_order = ["Active", "Passive", "Control"]
label_order = ["Control", "Schizophrenia"]


ERP columns available for focused EDA: ['Fz_N100_avg', 'FCz_N100_avg', 'Cz_N100_avg', 'Fz_P200_avg', 'FCz_P200_avg', 'Cz_P200_avg', 'Fz_B0_avg', 'FCz_B0_avg', 'Cz_B0_avg']


### **Age Distribution by Class**

In [27]:
fig = px.histogram(
    subject_demo_pdf,
    x="age",
    facet_col="label_name",   
    nbins=10,
    histnorm="percent",
    color="label_name",
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={"label_name": label_order}
)

fig = apply_layout(
    fig,
    "Age Distribution by Class",
    "Age",
    "Percentage"
)

fig.update_layout(showlegend=False)

fig.for_each_annotation(
    lambda a: a.update(
        text=f"<b>{a.text.split('=')[-1]}</b>",
        font=dict(color="black")
    )
)

fig.update_traces(
    opacity=0.9,
    marker_line_color="white",
    marker_line_width=1
)

fig.show()

### **Interpretation**

The age distributions of control and schizophrenia groups show a generally similar range, spanning from early adulthood to around 65 years. 

However, control subjects appear more concentrated in the younger age ranges (around early 20s), while schizophrenia patients are more evenly distributed across a wider range of ages, including a slightly higher presence in middle age.

Overall, there is no strong separation in age between the two groups, suggesting that age alone is unlikely to be a distinguishing factor for classification. This indicates that the model will need to rely more on EEG-derived features rather than demographic differences.

### **Education Distribution by Class**

In [31]:
fig = px.violin(
    subject_demo_pdf,
    x="label_name",
    y="education",
    color="label_name",
    points="all",
    box=False,
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={"label_name": label_order},
)
fig = apply_layout(fig, "Education Distribution by Class", "Class", "Years of Education")
fig.update_traces(meanline_visible=True, opacity=0.9, jitter=0.08, marker_size=4)
fig.show()


### **Interpretation**

The distribution of years of education differs slightly between the two groups. 

Control subjects tend to have higher and more concentrated education levels, with most values clustered around 15–18 years. 

In contrast, schizophrenia patients show a wider spread with a lower central tendency, indicating generally fewer years of education and greater variability.

This suggests that education may have some discriminative value, although the overlap between the two groups indicates it is not a strong standalone predictor and should be considered alongside other features.

In [47]:
GENDER_COLOR_MAP = {
    "M": "#44A194",
    "F": "#F7AD45"
}

fig = px.bar(
    gender_pct_pdf,
    x="label_name",
    y="percentage",
    color="gender",
    barmode="group",
    text=gender_pct_pdf["percentage"].astype(str) + "%",
    color_discrete_map=GENDER_COLOR_MAP,
    category_orders={"label_name": label_order},
)

fig = apply_layout(
    fig,
    "Gender Distribution by Class",
    "Class",
    "Percentage of Subjects"
)

fig.update_traces(
    textposition="outside",
    marker_line_color="white",
    marker_line_width=1.1
)

fig.update_yaxes(showticklabels=False)

fig.show()

### **Interpretation**

The gender distribution is highly imbalanced in both classes, with males representing the majority in both control (≈81%) and schizophrenia (≈84%) groups. The proportions are very similar across the two classes, indicating that gender is not a distinguishing factor between them in this dataset.

This suggests that gender is unlikely to contribute meaningful predictive signal for classification and may have limited value as a feature compared to EEG-derived variables.

### **N100 and P200 Distribution at Frontal and Central Electrodes**

##### **Question:** Do brain responses (N100 & P200) differ between Control and Schizophrenia?

`Definitions:`

- **N100** features represent early brain response ~100ms. They measure early auditory/sensory processing.

- **P200** features represent later response ~200ms. They measure higher-level processing / attention.

In [44]:
summary_pdf = (
    erp_shape_pdf
    .groupby(["label_name", "feature_label"], as_index=False)
    .agg(
        mean_value=("value", "mean"),
        std_value=("value", "std")
    )
)

fig = px.scatter(
    summary_pdf,
    x="feature_label",
    y="mean_value",
    color="label_name",
    error_y="std_value",
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={"label_name": label_order},
)

fig = apply_layout(
    fig,
    "Mean ERP Response by Class",
    "Feature",
    "Mean Amplitude"
)

fig.update_traces(marker=dict(size=11))

fig.show()

### **Interpretation**

This plot compares the **mean ERP amplitudes** (N100 and P200) across key frontocentral electrodes for control and schizophrenia groups, with error bars indicating variability.

Overall, the mean values for both groups are very close across all features, with only small differences in amplitude. For N100 (negative component), schizophrenia patients tend to show slightly less negative values (closer to zero), while for P200 (positive component), the differences between groups are minimal.

However, the large error bars indicate high variability within both groups, and the strong overlap suggests that these features alone do not clearly separate the two classes. 

This implies that individual ERP amplitudes may have limited discriminative power on their own and should be combined with other features (such as suppression effects or engineered features) for better classification performance.

`Definition:`

- ERP amplitude = how strong the brain’s response is at a specific moment in time

In [54]:
# Reuse eda_plot_pdf and label_order from earlier cells

relationship_specs = [
   ("FCz_N100_avg", "FCz_P200_avg", "FCz"),
    ("Fz_N100_avg", "Fz_P200_avg", "Fz"),
    ("Cz_N100_avg", "Cz_P200_avg", "Cz"),
]

valid_specs = [
    (x_col, y_col, title)
    for x_col, y_col, title in relationship_specs
    if x_col in eda_plot_pdf.columns and y_col in eda_plot_pdf.columns
]

subject_rel_pdf = (
    eda_plot_pdf[
        ["subject", "label_name"] +
        sorted({c for x_col, y_col, _ in valid_specs for c in (x_col, y_col)})
    ]
    .groupby(["subject", "label_name"], as_index=False)
    .mean(numeric_only=True)
)

fig = make_subplots(
    rows=1,
    cols=len(valid_specs),
    subplot_titles=[title for _, _, title in valid_specs],
    horizontal_spacing=0.08,
)

for col_idx, (x_col, y_col, _) in enumerate(valid_specs, start=1):
    pair_pdf = subject_rel_pdf[[x_col, y_col, "label_name"]].dropna()

    for label_name in label_order:
        label_pdf = pair_pdf[pair_pdf["label_name"] == label_name]

        fig.add_trace(
            go.Scatter(
                x=label_pdf[x_col],
                y=label_pdf[y_col],
                mode="markers",
                name=label_name,
                legendgroup=label_name,
                showlegend=(col_idx == 1),
                marker=dict(
                    color=CLASS_COLOR_MAP[label_name],
                    size=10,
                    opacity=0.75,
                    line=dict(color="white", width=0.7),
                ),
                hovertemplate=(
                    f"{x_col}: %{{x:.2f}}<br>"
                    f"{y_col}: %{{y:.2f}}"
                    f"<extra>{label_name}</extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )

fig = apply_layout(fig, "Subject-Level ERP Feature Relationships", None, None)

fig.for_each_annotation(
    lambda a: a.update(text=f"<b>{a.text}</b>", font=dict(color="black"))
)

for i, (x_col, y_col, _) in enumerate(valid_specs, start=1):
    fig.update_xaxes(title_text=x_col.replace("_avg", ""), row=1, col=i, color="black")
    fig.update_yaxes(title_text=y_col.replace("_avg", ""), row=1, col=i, color="black")

fig.show()


### **Interpretation**

This plot examines the relationship between early (N100) and later (P200) ERP responses at the subject level across key electrodes.

Across all three electrodes (Fz, FCz, Cz), there is a general positive relationship between N100 and P200, indicating that subjects with stronger early responses tend to also show stronger later responses. 

However, the distributions of control and schizophrenia subjects largely overlap in all subplots, with no clear separation or distinct clustering between the two groups.

Schizophrenia subjects appear to exhibit slightly greater variability, particularly in P200 amplitudes, but this difference is not strong enough to provide clear class separation. 

Overall, this suggests that the relationship between N100 and P200 alone is not sufficient to distinguish between groups, and more informative features, such as condition-based suppression effects, are likely required for effective classification.

### **Condition Effects on N100 Features**

In [ ]:
condition_effect_features = [c for c in ["Fz_N100_avg", "FCz_N100_avg"] if c in eda_plot_pdf.columns]

condition_effect_pdf = eda_plot_pdf[["condition_name", "label_name", *condition_effect_features]].melt(
    id_vars=["condition_name", "label_name"],
    value_vars=condition_effect_features,
    var_name="feature",
    value_name="value",
)

condition_effect_pdf["feature_label"] = condition_effect_pdf["feature"].str.replace("_avg", "", regex=False)

fig = px.violin(
    condition_effect_pdf,
    x="condition_name",
    y="value",
    color="label_name",
    facet_col="feature_label",
    points=False,   
    box=True,      
    violinmode="group",
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={
        "condition_name": condition_order,
        "label_name": label_order
    },
)

fig = apply_layout(
    fig,
    "Condition Effects on N100 Features",
    None,              
    "Feature Value"
)

fig.for_each_annotation(
    lambda a: a.update(
        text=f"<b>{a.text.split('=')[-1]}</b>",
        font=dict(color="black")
    )
)

fig.update_traces(
    meanline_visible=True,
    opacity=0.78
)

fig.update_xaxes(title_text="")

fig.add_annotation(
    text="Condition",
    x=0.5,
    y=-0.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16, color="black")
)

fig.show()

### **Interpretation**

This plot shows the distribution of N100 amplitudes across different conditions (Active, Passive, Control) for both classes.

Across both Fz and FCz electrodes, the overall distributions for control and schizophrenia groups largely overlap, indicating that raw N100 amplitudes within each condition are not strongly discriminative on their own. 

However, subtle shifts can be observed, where control subjects tend to exhibit slightly more negative N100 values compared to schizophrenia patients.

More importantly, the spread and variability appear larger in the schizophrenia group, suggesting less consistent neural responses. 

While condition-specific amplitudes alone do not clearly separate the classes, they provide the foundation for analyzing suppression effects between conditions.

### **Condition-Based N100 Responses (Active vs Passive) by Class**

In [58]:
suppression_mean_sdf = (
    eda_df
    .filter(F.col("condition").isin([1, 2]))
    .groupBy("subject", "label", "condition")
    .agg(
        F.avg("Fz_N100_avg").alias("Fz_N100_avg"),
        F.avg("FCz_N100_avg").alias("FCz_N100_avg"),
    )
)

suppression_long_pdf = suppression_mean_sdf.toPandas()

suppression_long_pdf["label_name"] = suppression_long_pdf["label"].map(LABEL_NAME_MAP)
suppression_long_pdf["condition_name"] = suppression_long_pdf["condition"].map(CONDITION_NAME_MAP)

suppression_compare_pdf = suppression_long_pdf.melt(
    id_vars=["subject", "label_name", "condition_name"],
    value_vars=[c for c in ["Fz_N100_avg", "FCz_N100_avg"] if c in suppression_long_pdf.columns],
    var_name="feature",
    value_name="value",
)

suppression_compare_pdf["feature_label"] = suppression_compare_pdf["feature"].str.replace("_avg", "", regex=False)

fig = px.strip(
    suppression_compare_pdf,
    x="condition_name",
    y="value",
    color="label_name",
    facet_col="feature_label",
    stripmode="group",
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={
        "condition_name": ["Active", "Passive"],
        "label_name": label_order
    }
)

fig = apply_layout(
    fig,
    "Active vs Passive N100 Response by Class",
    None,
    "Subject-Level Mean N100 (Fz / FCz)"
)

fig.update_traces(
    opacity=0.68,
    marker=dict(
        size=7,
        line=dict(color="white", width=0.6)
    )
)

fig.for_each_annotation(
    lambda a: a.update(
        text=f"<b>{a.text.split('=')[-1]}</b>",
        font=dict(color="black")
    )
)

fig.update_xaxes(title_text="")

fig.add_annotation(
    text="Condition",
    x=0.5,
    y=-0.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16, color="black")
)

fig.show()

### **Subject-level N100 Supression Scores**

In [ ]:
suppression_wide_pdf = suppression_long_pdf.pivot_table(
    index=["subject", "label_name"],
    columns="condition_name",
    values=[c for c in ["Fz_N100_avg", "FCz_N100_avg"] if c in suppression_long_pdf.columns],
)

suppression_score_rows = []
for feature in ["Fz_N100_avg", "FCz_N100_avg"]:
    if (feature, "Passive") in suppression_wide_pdf.columns and (feature, "Active") in suppression_wide_pdf.columns:
        feature_df = suppression_wide_pdf[[(feature, "Passive"), (feature, "Active")]].copy()
        feature_df.columns = ["passive", "active"]
        feature_df = feature_df.reset_index()
        feature_df["suppression_score"] = feature_df["passive"] - feature_df["active"]
        feature_df["feature_label"] = feature.replace("_avg", "")
        suppression_score_rows.append(feature_df)

suppression_score_pdf = pd.concat(suppression_score_rows, ignore_index=True)

fig = px.violin(
    suppression_score_pdf,
    x="label_name",
    y="suppression_score",
    color="label_name",
    facet_col="feature_label",
    points="all",
    box=False,
    color_discrete_map=CLASS_COLOR_MAP,
    category_orders={"label_name": label_order},
)
fig = apply_layout(fig, "Subject-Level N100 Suppression Scores (Passive - Active)", "Class", "Suppression Score")
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>", font=dict(color="black")))
fig.update_traces(meanline_visible=True, opacity=0.78, jitter=0.05, marker_size=4)

fig.update_xaxes(title_text="")

fig.add_annotation(
    text="Condition",
    x=0.5,
    y=-0.12,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16, color="black")
)

fig.show()

26/04/14 11:28:04 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 979958 ms exceeds timeout 120000 ms
26/04/14 11:28:04 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/14 11:44:02 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o

### **Interpretation**

The plots compare subject-level N100 responses between Active and Passive conditions for control and schizophrenia groups across Fz and FCz electrodes.

In simple terms, **suppression** refers to how much the brain reduces its response when a stimulus is self-generated (Active) compared to when it is externally generated (Passive). A healthy brain typically shows a weaker response in the Active condition because it can predict the stimulus.

For control subjects, Passive responses are generally more negative than Active responses, indicating a clear suppression effect. This suggests that the brain is successfully reducing its response to expected, self-generated stimuli.

In contrast, schizophrenia subjects show a smaller and less consistent difference between Active and Passive responses, with significant overlap between the two conditions. This indicates reduced suppression, meaning the brain is not effectively distinguishing between self-generated and external stimuli.

Overall, this pattern suggests that impaired suppression is a key difference in schizophrenia and motivates the use of suppression-based features for classification.